In [20]:
import os

import pandas as pd
import torch
from transformers import pipeline
from tqdm import tqdm

In [21]:
device = 0 if torch.backends.mps.is_available() else -1

In [22]:
classifier = pipeline(
    "text-classification", 
    model="bhadresh-savani/distilbert-base-uncased-emotion", 
    top_k=None,
    device=device,
    token=os.getenv("HF_TOKEN")
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [23]:
def get_emotion_vector(text):
    clean_text = str(text)[:512]
    results = classifier(clean_text)
    return [res['score'] for res in results[0]]

In [24]:
df = pd.read_csv("../../../data/cleaned_data.csv")
features = []
for text in tqdm(df['cleaned_statement']):
    features.append(get_emotion_vector(text))

100%|██████████| 31993/31993 [04:36<00:00, 115.53it/s]


In [25]:
emotion_cols = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
features_df = pd.DataFrame(features, columns=emotion_cols)
features_df['label'] = df['label']

In [26]:
features_df.to_csv('../../../data/features_for_model.csv', index=False)